## Data Normalization

In [5]:
import pandas as pd
import numpy as np
import re
import unicodedata
import os

In [6]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)
    
RAW_DIR  = '../student_resource/dataset/train/' 
PROC_DIR = '../student_resource/processed'

os.makedirs(PROC_DIR, exist_ok=True)

In [ ]:
s1 = pd.read_csv(f'{RAW_DIR}/train_source1.tsv', sep='\t')
s2 = pd.read_csv(f'{RAW_DIR}/train_source2.tsv', sep='\t')
s3 = pd.read_csv(f'{RAW_DIR}/train_source3.tsv', sep='\t')
print('shapes:', s1.shape, s2.shape, s3.shape)

shapes: (2206821, 4) (5034616, 4) (5285603, 4)


### Cleaning
1. Accent stripping

In [8]:
def strip_accents(text):
    if not isinstance(text, str):
        return ''
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

strip_accents('Caf\u00e9 M\u00fcller \u2014 fa\u00e7ade \u2116 12'), strip_accents('stra\u00dfe')

('Cafe Muller  facade No 12', 'strae')

2. Name normalization

In [10]:
NAME_ABBREV = {
    'incorporated': 'inc', 'inc': 'inc',
    'corporation': 'corp', 'corp': 'corp',
    'limited': 'ltd', 'ltd': 'ltd',
    'company': 'co', 'co': 'co',
    'private': 'pvt', 'pvt': 'pvt',
    'international': 'intl', 'intl': 'intl',
    'enterprises': 'ent', 'ent': 'ent',
    'services': 'svc', 'svc': 'svc',
}

#droping generic words fro blocking keys
LEGAL_SUFFIXES = {'inc', 'corp', 'ltd', 'llc', 'co', 'pvt', 'ent', 'svc',
                  'industries', 'ind', 'group', 'grp', 'holdings', 'hldg'}

def clean_name(name, drop_legal_suffix=False):
    if not isinstance(name, str) or not name.strip():
        return 
    t = strip_accents(name).lower()
    t = t.replace('&', ' and ')
    t = re.sub(r"[^a-z0-9\\s]", ' ', t) 
    tokens = [NAME_ABBREV.get(w, w) for w in t.split()]
    if drop_legal_suffix:
        tokens = [w for w in tokens if w not in LEGAL_SUFFIXES]
    return ' '.join(tokens)

  

Testing!

In [10]:
for raw in ["MCDONALD'S CORPORATION", "McDonald's Corp.",
            'Caf\u00e9 & Bar Pvt. Ltd.', 'Tata Consultancy Services Ltd',
            'STARBUCKS & CO', 'starbucks company']:
    print(f'{raw!r:40} -> {clean_name(raw)!r:32} | blocking: {clean_name(raw, True)!r}')

"MCDONALD'S CORPORATION"                 -> 'mcdonald s corp'                | blocking: 'mcdonald s'
"McDonald's Corp."                       -> 'mcdonald s corp'                | blocking: 'mcdonald s'
'Café & Bar Pvt. Ltd.'                   -> 'cafe and bar pvt ltd'           | blocking: 'cafe and bar'
'Tata Consultancy Services Ltd'          -> 'tata consultancy svc ltd'       | blocking: 'tata consultancy'
'STARBUCKS & CO'                         -> 'starbucks and co'               | blocking: 'starbucks and'
'starbucks company'                      -> 'starbucks co'                   | blocking: 'starbucks'


3. Addresses

In [11]:
ADDR_ABBREV = {
    "street": "st", "st": "st",
    "road": "rd", "rd": "rd",
    "avenue": "ave", "ave": "ave",
    "boulevard": "blvd", "blvd": "blvd",
    "drive": "dr", "dr": "dr",
    "lane": "ln", "ln": "ln",
    "floor": "fl", "suite": "ste", "unit": "ste",
    "apartment": "apt", "apt": "apt",
    "north": "n", "south": "s", "east": "e", "west": "w",
    "no": "", "number": "",   # 'No 12' -> '12'
}

LANDMARK_WORDS = {"near", "opp", "opposite", "behind", "beside",
                  "adjacent", "above", "below"}

def clean_address(addr):
    if not isinstance(addr, str) or not addr.strip():
        return ""
    t = strip_accents(addr).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    tokens = [ADDR_ABBREV.get(w, w) for w in t.split()]
    return " ".join(w for w in tokens if w)

def drop_landmark_tail(addr_clean):
    """Landmark refs are unmatchable noise."""
    tokens = addr_clean.split()
    for i, w in enumerate(tokens):
        if w in LANDMARK_WORDS:
            return " ".join(tokens[:i])
    return addr_clean

def extract_house_number(addr_clean):
    """First standalone number = house number"""
    m = re.search(r"\b(\d{1,6})\b", addr_clean or "")
    return m.group(1) if m else ""

def extract_postal_code(addr_clean, country):
    """US: 5-digit ZIP. India: 6-digit PIN. France: 5-digit"""
    if not addr_clean:
        return ""
    if str(country).upper() == "US":
        m = re.search(r"\b(\d{5})(?:-\d{4})?\b", addr_clean)
    else:
        m = re.search(r"\b(\d{5,6})\b", addr_clean)
    return m.group(1) if m else ""

In [19]:
def normalize_dataframe(df):
    """Add normalized columns"""
    df = df.copy()
    df["country_norm"] = df["country"].fillna("").astype(str).str.strip().str.lower()

    df["name_clean"]    = df["business_name"].apply(clean_name)
    df["name_blocking"] = df["business_name"].apply(lambda x: clean_name(x, drop_legal_suffix=True))
    df["name_tokens"]   = df["name_blocking"].fillna("").astype(str).apply(str.split)
    df["name_sort"]     = df["name_blocking"].apply(
        lambda x: " ".join(sorted(x.split())) if isinstance(x, str) else x)

    df["addr_clean"]  = df["business_address"].apply(clean_address).apply(drop_landmark_tail)
    df["addr_house"]  = df["addr_clean"].apply(extract_house_number)
    df["addr_postal"] = [extract_postal_code(a, c)
                         for a, c in zip(df["addr_clean"], df["country_norm"])]
    return df

In [20]:
s1n = normalize_dataframe(s1)
s2n = normalize_dataframe(s2)
s3n = normalize_dataframe(s3)
print('Normalized:', s1n.shape, s2n.shape, s3n.shape)

Normalized: (2206821, 12) (5034616, 12) (5285603, 12)


In [21]:
cols = ['business_name', 'name_clean', 'name_blocking',
        'business_address', 'addr_clean', 'addr_house', 'addr_postal']
s2n[cols].sample(8, random_state=42)

,business_name,name_clean,name_blocking,business_address,addr_clean,addr_house,addr_postal
2641297,Harbor Center,harbor center,harbor center,"6309 EVANGELINE TRAIL, AUSTIN, TX",6309 evangeline trail austin tx,6309,
1856454,STEWARD ACE PREMIER,steward ace premier,steward ace premier,"1029 MAPLE HILL RD, LEBANON, TN",1029 maple hill rd lebanon tn,1029,
371269,Klapper and Lott,klapper and lott,klapper and lott,"368 VINE STREET, TOOELE, UT",368 vine st tooele ut,368,
793182,EAR NOSE & THROAT CARE,ear nose and throat care,ear nose and throat care,"YORK RD, LUTHERVILLE, MD",york rd lutherville md,,
4159519,Shield Inrfabuild Private Limited,shield inrfabuild pvt ltd,shield inrfabuild,"206 FLAT NO. LG 1 KH NO. 514, 516 SHYAM BHAWAN JHANDA COLONY FATEHPUR, DELHI, Delhi",206 flat lg 1 kh 514 516 shyam bhawan jhanda colony fatehpur delhi delhi,206,
2290871,#holtline,holtline,holtline,"HOUSTON, 6119 DARLINGHURST DRIVE, TX",houston 6119 darlinghurst dr tx,6119,
2479261,Cancer [Services],cancer svc,cancer,NaN,,,
606462,Smt Hermes Private Limited Center,smt hermes pvt ltd center,smt hermes center,"Delhi, HOUSE NO 138, 2ND FLOOR, BLK C, PKT 2 DDA MARKET, SECTOR 16, ROHINI, DELHI",delhi house 138 2nd fl blk c pkt 2 dda market sector 16 rohini delhi,138,


In [ ]:
def load_and_normalize(path):
    """read tsv -> normalize -> return"""
    df = pd.read_csv(path, sep="\t")
    return normalize_dataframe(df)

In [23]:
for name, df in [('S1', s1n), ('S2', s2n), ('S3', s3n)]:
    print(f'{name}')
    print(f'  empty name_clean: {(df["name_clean"] == "").sum()}')
    print(f'  empty addr_clean: {(df["addr_clean"] == "").sum()}')
    print(f'  house found    : {(df["addr_house"] != "").mean()*100:.1f}%')
    print(f'  postal found   : {(df["addr_postal"] != "").mean()*100:.1f}%')

exact = set(zip(s2n['name_sort'], s2n['country_norm'])) & \
    set(zip(s1n['name_sort'], s1n['country_norm']))

print(f'\\nExact (name_sort+country) collisions S1-S2: {len(exact):,} near-certain pairs for free')

S1
  empty name_clean: 0
  empty addr_clean: 2743
  house found    : 94.0%
  postal found   : 6.6%
S2
  empty name_clean: 455759
  empty addr_clean: 172466
  house found    : 87.3%
  postal found   : 6.5%
S3
  empty name_clean: 243630
  empty addr_clean: 180197
  house found    : 87.4%
  postal found   : 6.5%
\nExact (name_sort+country) collisions S1-S2: 963,164 near-certain pairs for free


Comparing with Ground Truth
- TRUE matched pairs actually become similar after normalization

In [24]:
gt = pd.read_csv(F"{RAW_DIR}/train_ground_truth.tsv", sep="\t")

In [25]:
s1_idx  = s1n.set_index('entity_id')
s23_idx = pd.concat([s2n, s3n]).set_index('entity_id')
name_hits = postal_hits = checked = 0
for _, row in gt.head(5000).iterrows():
    s1id = row['source1_entity_id']
    if s1id not in s1_idx.index:
        continue
    a = s1_idx.loc[s1id]
    for mid in str(row['matched_entity_ids']).split(','):
        mid = mid.strip()
        if mid and mid in s23_idx.index:
            b = s23_idx.loc[mid]
            name_hits   += int(a['name_sort'] == b['name_sort'] and a['name_sort'] != '')
            postal_hits += int(a['addr_postal'] != '' and a['addr_postal'] == b['addr_postal'])

In [27]:
print(f'checked {checked:,} true pairs:')
print(f'  identical name_sort : {name_hits/(checked+1)*100:.1f}%')
print(f'  identical postal    : {postal_hits/(checked+1)*100:.1f}%')
print('Highest is PRIMARY blocking key')

checked 0 true pairs:
  identical name_sort : 847900.0%
  identical postal    : 78500.0%
Highest is PRIMARY blocking key


In [28]:
def dump_normalized(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    out = df.copy()
    if 'name_tokens' in out.columns:           
        out['name_tokens'] = out['name_tokens'].apply(
            lambda toks: ' '.join(toks) if isinstance(toks, list) else '')
    out.to_csv(path, sep='\t', index=False, encoding='utf-8')

def load_normalized(path):
    df = pd.read_csv(path, sep='\t')          # TAB in, TAB out
    if 'name_tokens' in df.columns:           # 'a b c' -> ['a','b','c']
        df['name_tokens'] = df['name_tokens'].fillna('').apply(str.split)
    return df

In [29]:
# dump all three sources in one go

for key, df in [('1', s1n), ('2', s2n), ('3', s3n)]:
    dump_normalized(df, f'{PROC_DIR}/train_source{key}_normalized.tsv')
print('dumped:', os.listdir(PROC_DIR))

dumped: ['train_source1_normalized.tsv', 'train_source2_normalized.tsv', 'train_source3_normalized.tsv']


#### Round Trip test

In [30]:
check = load_normalized(f'{PROC_DIR}/train_source2_normalized.tsv')

In [ ]:
assert len(check) == len(s2n), 'rows lost!'
assert (check['name_blocking'] == s2n['name_blocking']).all(), 'name_blocking changed!'
assert (check['addr_postal'].fillna('') == s2n['addr_postal'].fillna('')).all()


In [ ]:
print('round-trip -> rows:', len(check))

round-trip -> rows: 5034616 | sample tokens: []


normalization done

- `name_clean` / `name_blocking` / `name_sort` (features + blocking)
- `addr_clean` / `addr_house` / `addr_postal` address keys for blocking
- Accent-stripping ready for France \ud83c\uddeb\ud83c\uddf7, safe fallback for unknown countries
- Raw columns untouched 
- Clean dump/load 
